# 01 Prepare RegGS Scene (Re10k-1)

Goal:

1. adapt `Re10k-1` into a RegGS-ready scene;
2. preserve the **full sequence**;
3. prefer **symlink over copy** when images do not need pixel changes;
4. export clean `intrinsics.json` and `cameras.json`;
5. verify the exported scene layout.

This notebook is the first concrete Part 2 adapter and uses the canonical prepared-data path:

```text
dataset/Re10k-1/part2/<scene_name>/
```


In [1]:
from pathlib import Path
import json
import os
import shutil
import struct

CV_ROOT = Path('~/CV_Project').expanduser()
DATASET_ROOT = CV_ROOT / 'dataset'

SOURCE_ROOT = DATASET_ROOT / 'Re10k-1'
SOURCE_IMAGE_DIR = SOURCE_ROOT / 'images'
SOURCE_CAMERAS_JSON = SOURCE_ROOT / 'cameras.json'
PART2_ROOT = SOURCE_ROOT / 'part2'

PART2_ROOT.mkdir(parents=True, exist_ok=True)

print('SOURCE_ROOT =', SOURCE_ROOT)
print('PART2_ROOT =', PART2_ROOT)


SOURCE_ROOT = /home/bzhang512/CV_Project/dataset/Re10k-1
PART2_ROOT = /home/bzhang512/CV_Project/dataset/Re10k-1/part2


## 1. Export configuration


In [2]:
EXPORT_SCENE_NAME = 'reggs_re10k1_fullseq_256'
EXPORT_ROOT = PART2_ROOT / EXPORT_SCENE_NAME
EXPORT_IMAGE_DIR = EXPORT_ROOT / 'images'

# Re10k-1 source images are already 256x256, so the default path can use symlinks.
# If you later change target size, set TARGET_SIZE to (W, H) and the notebook will write resized images instead.
TARGET_SIZE = None
OVERWRITE_EXPORT = True

print('EXPORT_ROOT =', EXPORT_ROOT)
print('TARGET_SIZE =', TARGET_SIZE)
print('OVERWRITE_EXPORT =', OVERWRITE_EXPORT)


EXPORT_ROOT = /home/bzhang512/CV_Project/dataset/Re10k-1/part2/reggs_re10k1_fullseq_256
TARGET_SIZE = None
OVERWRITE_EXPORT = True


## 2. Helpers


In [4]:
def read_png_size(path: Path):
    png_sig = bytes.fromhex('89504E470D0A1A0A')
    with path.open('rb') as f:
        sig = f.read(8)
        if sig != png_sig:
            raise ValueError(f'Not a PNG file: {path}')
        _length = struct.unpack('>I', f.read(4))[0]
        chunk_type = f.read(4)
        if chunk_type != b'IHDR':
            raise ValueError(f'Invalid PNG header: {path}')
        width, height = struct.unpack('>II', f.read(8))
    return width, height


def safe_reset_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def symlink_file(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    os.symlink(src, dst)


def load_re10k_source():
    cameras = json.loads(SOURCE_CAMERAS_JSON.read_text(encoding='utf-8'))
    cameras = sorted(cameras, key=lambda x: x['image_name'])
    frame_paths = [SOURCE_IMAGE_DIR / cam['image_name'] for cam in cameras]
    missing = [str(p) for p in frame_paths if not p.exists()]
    if missing:
        raise FileNotFoundError(f'Missing source images: {missing[:5]}')

    width, height = read_png_size(frame_paths[0])

    fx_values = {round(cam['fx'], 10) for cam in cameras}
    fy_values = {round(cam['fy'], 10) for cam in cameras}
    cx_values = {round(cam['cx'], 10) for cam in cameras}
    cy_values = {round(cam['cy'], 10) for cam in cameras}

    if not (len(fx_values) == len(fy_values) == len(cx_values) == len(cy_values) == 1):
        raise ValueError('Re10k-1 currently appears to have varying normalized intrinsics across frames.')

    intrinsics = {
        'fx': cameras[0]['fx'],
        'fy': cameras[0]['fy'],
        'cx': cameras[0]['cx'],
        'cy': cameras[0]['cy'],
    }

    return {
        'cameras': cameras,
        'frame_paths': frame_paths,
        'source_width': width,
        'source_height': height,
        'intrinsics': intrinsics,
    }


source = load_re10k_source()
print('num frames =', len(source['frame_paths']))
print('source size =', (source['source_width'], source['source_height']))
print('normalized intrinsics =', source['intrinsics'])


num frames = 279
source size = (256, 256)
normalized intrinsics = {'fx': 0.8572737166601563, 'fy': 0.857483078, 'cx': 0.5, 'cy': 0.5}


## 3. Inspect source scene


In [5]:
print('first five source frames:')
for p in source['frame_paths'][:5]:
    print(' -', p.name)

print()
print('first camera entry:')
print(json.dumps(source['cameras'][0], indent=2))


first five source frames:
 - 00000.png
 - 00001.png
 - 00002.png
 - 00003.png
 - 00004.png

first camera entry:
{
  "cam_id": 0,
  "cam_quat": [
    0.0003909160732291639,
    0.005096666980534792,
    0.0008775556343607605,
    0.9999865889549255
  ],
  "cam_trans": [
    0.027700699865818024,
    -0.009711414575576782,
    0.34730884432792664
  ],
  "cx": 0.5,
  "cy": 0.5,
  "fx": 0.8572737166601563,
  "fy": 0.857483078,
  "image_name": "00000.png",
  "timestamp": 45979267
}


## 4. Decide export mode


In [6]:
SOURCE_SIZE = (source['source_width'], source['source_height'])
USE_SYMLINKS = (TARGET_SIZE is None) or (TARGET_SIZE == SOURCE_SIZE)

if TARGET_SIZE is None:
    EXPORT_WIDTH, EXPORT_HEIGHT = SOURCE_SIZE
else:
    EXPORT_WIDTH, EXPORT_HEIGHT = TARGET_SIZE

print('SOURCE_SIZE =', SOURCE_SIZE)
print('EXPORT_SIZE =', (EXPORT_WIDTH, EXPORT_HEIGHT))
print('USE_SYMLINKS =', USE_SYMLINKS)


SOURCE_SIZE = (256, 256)
EXPORT_SIZE = (256, 256)
USE_SYMLINKS = True


## 5. Export Re10k-1 in RegGS scene format


In [7]:
def write_intrinsics_json(export_root: Path, intrinsics: dict):
    payload = {
        'fx': float(intrinsics['fx']),
        'fy': float(intrinsics['fy']),
        'cx': float(intrinsics['cx']),
        'cy': float(intrinsics['cy']),
    }
    (export_root / 'intrinsics.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')


def write_cameras_json(export_root: Path, cameras: list):
    export_cameras = []
    for i, cam in enumerate(cameras):
        export_cameras.append({
            'cam_id': int(i),
            'cam_quat': [float(x) for x in cam['cam_quat']],
            'cam_trans': [float(x) for x in cam['cam_trans']],
            'cx': float(cam['cx']),
            'cy': float(cam['cy']),
            'fx': float(cam['fx']),
            'fy': float(cam['fy']),
            'image_name': f'{i:04d}.png',
            'timestamp': cam.get('timestamp', i),
        })
    (export_root / 'cameras.json').write_text(json.dumps(export_cameras, indent=2), encoding='utf-8')


def export_re10k_scene(source: dict, export_root: Path, overwrite: bool = True):
    if export_root.exists() and not overwrite:
        raise FileExistsError(f'{export_root} already exists and overwrite=False')
    safe_reset_dir(export_root)
    image_dir = export_root / 'images'
    image_dir.mkdir(parents=True, exist_ok=True)

    if not USE_SYMLINKS:
        raise NotImplementedError('Resize/write mode is not implemented yet in this first Re10k notebook. Keep TARGET_SIZE=None for symlink export.')

    for i, src in enumerate(source['frame_paths']):
        dst = image_dir / f'{i:04d}.png'
        symlink_file(src, dst)

    write_intrinsics_json(export_root, source['intrinsics'])
    write_cameras_json(export_root, source['cameras'])

    return {
        'export_root': export_root,
        'num_frames': len(source['frame_paths']),
        'use_symlinks': USE_SYMLINKS,
        'export_size': (EXPORT_WIDTH, EXPORT_HEIGHT),
    }


summary = export_re10k_scene(source, EXPORT_ROOT, overwrite=OVERWRITE_EXPORT)
summary


{'export_root': PosixPath('/home/bzhang512/CV_Project/dataset/Re10k-1/part2/reggs_re10k1_fullseq_256'),
 'num_frames': 279,
 'use_symlinks': True,
 'export_size': (256, 256)}

## 6. Verify exported scene


In [8]:
export_cameras = json.loads((EXPORT_ROOT / 'cameras.json').read_text(encoding='utf-8'))
export_intrinsics = json.loads((EXPORT_ROOT / 'intrinsics.json').read_text(encoding='utf-8'))
export_images = sorted((EXPORT_ROOT / 'images').iterdir())

print('export root exists =', EXPORT_ROOT.exists())
print('num exported images =', len(export_images))
print('intrinsics =', export_intrinsics)
print('num camera entries =', len(export_cameras))
print('first exported camera =')
print(json.dumps(export_cameras[0], indent=2))

print()
print('first five exported image entries:')
for p in export_images[:5]:
    target = p.resolve() if p.is_symlink() else p
    print(f' - {p.name} -> {target}')


export root exists = True
num exported images = 279
intrinsics = {'fx': 0.8572737166601563, 'fy': 0.857483078, 'cx': 0.5, 'cy': 0.5}
num camera entries = 279
first exported camera =
{
  "cam_id": 0,
  "cam_quat": [
    0.0003909160732291639,
    0.005096666980534792,
    0.0008775556343607605,
    0.9999865889549255
  ],
  "cam_trans": [
    0.027700699865818024,
    -0.009711414575576782,
    0.34730884432792664
  ],
  "cx": 0.5,
  "cy": 0.5,
  "fx": 0.8572737166601563,
  "fy": 0.857483078,
  "image_name": "0000.png",
  "timestamp": 45979267
}

first five exported image entries:
 - 0000.png -> /home/bzhang512/CV_Project/dataset/Re10k-1/images/00000.png
 - 0001.png -> /home/bzhang512/CV_Project/dataset/Re10k-1/images/00001.png
 - 0002.png -> /home/bzhang512/CV_Project/dataset/Re10k-1/images/00002.png
 - 0003.png -> /home/bzhang512/CV_Project/dataset/Re10k-1/images/00003.png
 - 0004.png -> /home/bzhang512/CV_Project/dataset/Re10k-1/images/00004.png


## 7. Minimal RegGS compatibility check


In [9]:
assert (EXPORT_ROOT / 'images').exists()
assert (EXPORT_ROOT / 'intrinsics.json').exists()
assert (EXPORT_ROOT / 'cameras.json').exists()
assert len(export_images) == len(export_cameras) == len(source['frame_paths'])
assert export_images[0].name == '0000.png'
assert export_cameras[0]['image_name'] == '0000.png'
print('RegGS-ready scene check passed.')


RegGS-ready scene check passed.
